In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  STEP 0~4 : 연도별 합산 → NULL→0 → 전치 → 파생속성 생성         ║
# ║  build_master_A + preprocess_A 통합 버전                         ║
# ╚══════════════════════════════════════════════════════════════════╝
import csv, glob, os
import numpy as np
import pandas as pd
from collections import defaultdict

# ── 경로 설정 ──────────────────────────────────────────────────────
DATA_DIR   = "./data"                          # ← 원본 CSV 폴더
MASTER_OUT = "kangxi_master_A_1661_1722.csv"   # 중간 산출물
FINAL_OUT  = "kangxi_master_A_preprocessed.csv"  # 최종 전처리본

# ════════════════════════════════════════════════════════════════════
# [STEP 0]  연도별 단어 빈도 합산  →  NULL 포함 마스터 CSV 저장
# ════════════════════════════════════════════════════════════════════
print("=" * 55)
print("[STEP 0] 연도별 단어 빈도 합산 중...")
print("=" * 55)

files = sorted(glob.glob(f"{DATA_DIR}/*.csv"))
print(f"  총 파일 수: {len(files)}")

# data[단어][연도] = 합산 빈도
data       = defaultdict(lambda: defaultdict(int))
years_set  = set()
word_order = []
word_set   = set()

for f in files:
    fname = os.path.basename(f).replace(".csv", "")
    year  = fname[:4]           # 파일명 앞 4자리 = 연도
    years_set.add(year)

    with open(f, encoding="utf-8-sig", newline="") as fh:
        reader = csv.DictReader(fh)
        for row in reader:
            word     = row["한자단어"]
            freq_str = row["빈도"]
            try:
                freq = int(freq_str)
            except:
                freq = 0

            data[word][year] += freq

            if word not in word_set:
                word_set.add(word)
                word_order.append(word)

years_ordered = sorted(years_set)   # 1661 ~ 1722
print(f"  고유 단어 수 : {len(word_order):,}")
print(f"  연도 수     : {len(years_ordered)}")
print(f"  연도 범위   : {years_ordered[0]} ~ {years_ordered[-1]}")

# 저장 — 없는 연도는 NULL, 있으면 합산값
print("\n  CSV 저장 중...")
header = ["한자단어"] + years_ordered
with open(MASTER_OUT, "w", encoding="utf-8-sig", newline="") as fh:
    writer = csv.writer(fh)
    writer.writerow(header)
    for word in word_order:
        row = [word]
        for yr in years_ordered:
            val = data[word].get(yr)
            row.append("NULL" if val is None else str(val))
        writer.writerow(row)

print(f"  ✅ 저장 완료 → {MASTER_OUT}")
print(f"     shape : {len(word_order):,} 단어 × {len(years_ordered)} 연도\n")

# ════════════════════════════════════════════════════════════════════
# 전처리 공통 설정 : 제거할 한문 조사·허사 목록
# ════════════════════════════════════════════════════════════════════
PARTICLES = set([
    # 허사·어기조사
    "之","乎","者","也","而","以","於","其","為","與","乃","則","所","矣",
    "焉","哉","夫","且","若","如","兮","耳","爾","耶","邪","歟",
    # 부정·판단
    "不","無","非","莫","勿","毋","弗",
    # 지시·대명
    "此","彼","是","茲","斯","我","汝",
    # 연결·부사
    "皆","亦","即","各","又","更","已","故","因","遂","從","及",
    "或","仍","再","復","旋","頓","俱","並","并","尚","猶",
    "自","至","由","用","共","同","均","咸","悉",
    # 수량·정도
    "等","諸","餘","凡","每","盡","全","一","二","三","四",
    "五","六","七","八","九","十","百","千","萬","億","兩","半",
    # 방위·시간
    "上","下","前","後","中","內","外","左","右","東","西","南","北",
    "今","昔","先","初","末","始","終","早","晚","久","頃",
    # 동사성 허사
    "有","曰","云","言","稱","謂","令","使","命","允","欽","奉","遵",
    "奏","覆","議","准","照","依",
    # 단순 빈출 글자
    "大","小","多","少","新","舊","正","副","主","次","高","低",
    "都","公","行","事","日","月","年","時","處","地","人","王",
    "臣","民","兵","官","子","父","母","兄","弟","姪","孫",
    # 구두점
    "○","●","◎","△","□","■","▲","▼","◆","◇","☆","★",
    "·","、","。","，","：","；","！","？","「","」","『","』",
    "（","）","【","】","〔","〕","—","…","～",
    # 숫자 한자
    "零","壹","貳","參","肆","伍","陸","柒","捌","玖","拾","佰","仟",
    # 타깃 변수 (파생속성으로 따로 처리)
    "위기",
])

# ════════════════════════════════════════════════════════════════════
# [STEP 1]  원본 로드  (단어 × 연도, NULL 포함)
# ════════════════════════════════════════════════════════════════════
print("=" * 55)
print("[STEP 1] 마스터 CSV 로드 중...")
df = pd.read_csv(MASTER_OUT, encoding="utf-8-sig", index_col=0, dtype=str)
print(f"  원본 shape : {df.shape}  (단어 × 연도)")

# ════════════════════════════════════════════════════════════════════
# [STEP 2]  NULL → 0  (수치 변환)
# ════════════════════════════════════════════════════════════════════
print("\n[STEP 2] NULL → 0 변환...")
df_num = df.replace("NULL", "0").fillna("0").astype(float).astype(int)
print(f"  변환 후 shape : {df_num.shape}")

# ════════════════════════════════════════════════════════════════════
# [STEP 3]  전치  (연도 × 단어)
# ════════════════════════════════════════════════════════════════════
print("\n[STEP 3] 전치 (Transpose)...")
df_t = df_num.T                 # (62 연도) × (단어 수)
df_t.index.name = "연도"
print(f"  전치 후 shape : {df_t.shape}  (연도 × 단어)")

# ════════════════════════════════════════════════════════════════════
# [STEP 4]  의미없는 조사·허사 제거
# ════════════════════════════════════════════════════════════════════
print("\n[STEP 4] 의미없는 조사·허사·구두점 제거...")

keep_wigi  = df_t["위기"].copy()   # 위기 컬럼 미리 저장
remove_cols = [c for c in df_t.columns if c in PARTICLES]
df_t = df_t.drop(columns=remove_cols)

print(f"  제거된 단어 수 : {len(remove_cols):,}개")
print(f"  제거 샘플     : {remove_cols[:15]}")
print(f"  남은 단어 수  : {df_t.shape[1]:,}개")

# ════════════════════════════════════════════════════════════════════
# [STEP 5]  파생속성 생성  —  희귀단어(총합 ≤5) 를 양/음 상관으로 묶기
# ════════════════════════════════════════════════════════════════════
print("\n[STEP 5] 희귀단어 파생속성 생성 (총합 ≤5)...")

col_sums  = df_t.sum(axis=0)
rare_mask = col_sums <= 5
rare_cols = df_t.columns[rare_mask].tolist()
print(f"  총합 ≤5 희귀단어 수 : {len(rare_cols):,}개")

wigi_series = keep_wigi.astype(float)

pos_cols, neg_cols, nan_cols = [], [], []
for col in rare_cols:
    series = df_t[col].astype(float)
    if series.std() == 0 or wigi_series.std() == 0:
        nan_cols.append(col)
        continue
    cor = series.corr(wigi_series)
    if pd.isna(cor):
        nan_cols.append(col)
    elif cor > 0:
        pos_cols.append(col)
    else:
        neg_cols.append(col)

print(f"  양의 상관 희귀단어   : {len(pos_cols):,}개 → '희귀 단어 (양)' 합산")
print(f"  음의 상관 희귀단어   : {len(neg_cols):,}개 → '희귀 단어 (음)' 합산")
print(f"  상관 계산 불가(NaN)  : {len(nan_cols):,}개 → 제거")

# 파생속성 = 연도별 합산
df_t["희귀 단어 (양)"] = df_t[pos_cols].sum(axis=1)
df_t["희귀 단어 (음)"] = df_t[neg_cols].sum(axis=1)

# 희귀단어 원본 + NaN 컬럼 제거
drop_rare = rare_cols + nan_cols
df_t = df_t.drop(columns=[c for c in drop_rare if c in df_t.columns])

# 위기 컬럼 맨 마지막에 복원
df_t["위기"] = keep_wigi.astype(int)

print(f"\n  파생속성 추가 후 shape : {df_t.shape}  (연도 × 단어+파생)")

# ════════════════════════════════════════════════════════════════════
# [STEP 6]  잔존 NaN 제거 + 저장
# ════════════════════════════════════════════════════════════════════
print("\n[STEP 6] 잔존 NaN 제거 및 저장...")
null_count = df_t.isnull().sum().sum()
print(f"  잔존 NaN : {null_count}개 → fillna(0) 적용")
df_t = df_t.fillna(0).astype(int)

df_t.to_csv(FINAL_OUT, encoding="utf-8-sig")
size_mb = os.path.getsize(FINAL_OUT) / (1024 * 1024)

print(f"\n{'='*55}")
print(f"✅ 전처리 완료!")
print(f"  저장 위치  : {FINAL_OUT}")
print(f"  파일 크기  : {size_mb:.2f} MB")
print(f"  최종 shape : {df_t.shape}  (연도 × 피처)")
print(f"{'='*55}")

# ── 최종 검증 ─────────────────────────────────────────────────────
print("\n[검증] 마지막 5개 컬럼:")
print(f"  {df_t.columns[-5:].tolist()}")
print("\n[검증] 파생속성 + 위기 연도별 값:")
print(df_t[["희귀 단어 (양)", "희귀 단어 (음)", "위기"]].to_string())
